# Genizah v2.2a — extractive QA + documentary grounding + line-broken editions

Data lever: one **materialised, map-style** mixture (`isaacmg/genizah_v22_pilot`, images-once layout:
`rows/train.parquet`, `rows/val.parquet`, `images/<sha1>.jpg`) built by
`src/finetuning/qwen_hebrew/build_v22_mixture.py` from

| source | share | what |
|---|---|---|
| genizah_ktiv_v4 transcription | 0.30 | page / region / section / line transcription (KTIV) |
| pgp_editions_v1 | 0.25 | line-broken PGP edition pages + line-structure tasks |
| genizah_ktiv_v4 grounding | 0.15 | locate / read_box / line_index / detect / crop (KTIV geometry) |
| documentary_grounding_v1 | 0.10 | locate / read_box on two-reader agreed lines (Kraken geometry) |
| pgp_qa_v1 | 0.20 | extractive QA: dates, month/year, place, people, witnesses, parties (verbatim spans + line index) |

8,000 train rows (QA rows repeat ~1.8×), 200 val rows across every source. Map-style ⇒ no streaming, no
`DataLoaderDispatcher`, no stream-skip resume; the PREPARED-dataloader gate from v2.1b stays (patch rows vs
grid, prepared == direct loss). Warm start = **v2.1b step 1200** (flagship, `qwen3-vl-8b-heb-v21b-step1200`).
Everything else is v2.1b's: pinned install triplet, 6.5/7 MP resolution contract, LoRA r16 on tower +
language, merger FROZEN, adamw_8bit, cosine, hub checkpointing to a NEW repo.

Design and gates: `docs/v22_dataset.md`. Run name `genizah_v22a`; one name per run.


In [ ]:
# Cell 1 — installs + env (PINNED: the exact triplet audited 2026-08-09;
# unpinned installs float and transformers 5.x would invalidate the vision-path
# analysis AND the 108-tensor count)
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["WANDB_PROJECT"] = "qwen-hebrew-finetune"
%pip install -q "unsloth[colab-new]==2026.8.9" "unsloth_zoo==2026.8.6" "transformers==4.57.6" hf_transfer wandb

from importlib.metadata import version
assert version("unsloth_zoo") == "2026.8.6", f"unsloth_zoo drifted: {version('unsloth_zoo')}"
assert version("transformers") == "4.57.6", f"transformers drifted: {version('transformers')}"

import torch
assert torch.cuda.is_available(), "No GPU — switch runtime to A100"

In [ ]:
# Cell 2 — data: genizah_v22_pilot (images-once, MAP-STYLE, materialised mixture — no streaming)
import json
from collections import Counter
from pathlib import Path
from google.colab import userdata
from huggingface_hub import HfApi, login, snapshot_download

login(token=userdata.get("HF_TOKEN"))

V22_REPO = "isaacmg/genizah_v22_pilot"
V22_REVISION = "1a4f38b44c51c069301b488ca668a8013ae8c3b5"   # pushed 2026-09-25 00:16: 8,000 train / 188 val, 4,598 images ("PIN-AFTER-PUSH" would resolve main)
if V22_REVISION == "PIN-AFTER-PUSH":
    V22_REVISION = HfApi().dataset_info(V22_REPO).sha
    print(f"⚠️ unpinned: {V22_REPO} main -> {V22_REVISION} — pin this SHA before the next launch")
assert len(V22_REVISION) == 40, f"unpinned revision: {V22_REVISION!r}"
DATA_DIR = Path(snapshot_download(V22_REPO, repo_type="dataset", revision=V22_REVISION,
                                  local_dir="/content/genizah_v22_pilot"))
MIX = json.loads((DATA_DIR / "mixture.json").read_text())
print("mixture.json:", json.dumps(MIX, ensure_ascii=False)[:800])

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from datasets import Image as _HFImage
from torch.utils.data import Dataset as _TorchDataset

META_KEY = b"images_once"
SHA_COLUMN = "image_sha1"


class ImagesOnceDataset(_TorchDataset):
    """Map-style rows with images stored once (mirror of src/finetuning/qwen_hebrew/images_once.py).

    Items equal the original DatasetDict rows (image decoded, columns in the original order), so the
    v2.1b collator works unchanged. Rows stay in one Arrow table: forked DataLoader workers share it.
    """

    def __init__(self, rows_parquet, images_dir, tasks=None, check_files=True):
        table = pq.read_table(rows_parquet)
        meta = json.loads(table.schema.metadata[META_KEY])
        if tasks is not None:
            table = table.filter(pc.is_in(table["task"], value_set=pa.array(list(tasks))))
        self.images_dir = Path(images_dir)
        self.columns = meta["columns"]
        self.image_column = meta["image_column"]
        self._decoder = _HFImage(mode=meta["image_mode"])
        self._table = table.combine_chunks()
        if check_files:
            missing = [s for s in pc.unique(table[SHA_COLUMN]).to_pylist()
                       if not self.image_path(s).exists()]
            assert not missing, f"{len(missing)} images missing under {images_dir}, e.g. {missing[0]}.jpg"

    def __len__(self):
        return self._table.num_rows

    def image_path(self, sha1):
        return self.images_dir / f"{sha1}.jpg"

    def load_image(self, sha1):
        return self._decoder.decode_example({"path": None, "bytes": self.image_path(sha1).read_bytes()})

    def column(self, name):
        return self._table[name].to_pylist()

    def __getitem__(self, idx):
        n = len(self)
        if not -n <= idx < n:
            raise IndexError(f"row {idx} out of range for {n} rows")
        row = self._table.slice(idx % n, 1).to_pylist()[0]
        return {c: self.load_image(row[SHA_COLUMN]) if c == self.image_column else row[c]
                for c in self.columns}


train_ds = ImagesOnceDataset(DATA_DIR / "rows/train.parquet", DATA_DIR / "images")
eval_ds = ImagesOnceDataset(DATA_DIR / "rows/val.parquet", DATA_DIR / "images")
print(f"train rows={len(train_ds)} val rows={len(eval_ds)}")
assert len(train_ds) >= 7500 and len(eval_ds) >= 150, "pilot mixture incomplete"

_src = Counter(train_ds.column("source"))
_task = Counter(train_ds.column("task"))
_n = len(train_ds)
print("train by source:", {k: (v, round(v / _n, 3)) for k, v in _src.items()})
print("train by task:", dict(_task))
for _s, _lo in (("pgp_qa", 0.17), ("pgp_editions", 0.22), ("documentary_grounding", 0.08)):
    assert _src[_s] / _n >= _lo, f"{_s} share {_src[_s] / _n:.3f} below {_lo} — wrong dataset build"
assert set(eval_ds.column("source")) == set(_src), "val must cover every train source"

# Label hygiene (transcription answers) — same checks as v2.0a/v2.1b
_ans = train_ds.column("answer")
_q = train_ds.column("question")
_t = train_ds.column("task")
_srcs = train_ds.column("source")
_transc = [a for a, t in zip(_ans, _t) if t.endswith("transcribe")]
assert _transc and all("␣" not in a for a in _transc), "internal gap token leaked"
assert all(not any(ch in a for ch in "&#$_{}<>\\") for a in _transc), "mojibake leaked"
assert any("[...]" in a for a in _transc), "expected damage-gap markers"

# Grounding hygiene: valid 0-1000 boxes, self-describing prompts (KTIV + documentary)
def _ok_box(b):
    return len(b) == 4 and all(0 <= v <= 1000 for v in b) and b[2] > b[0] and b[3] > b[1]
_loc = [(a, q) for a, q, t in zip(_ans, _q, _t) if t in ("locate", "locate_word")]
assert _loc and all(_ok_box(json.loads(a)["bbox_2d"]) and "0-1000" in q for a, q in _loc), "bad locate box/prompt"
_doc = [(a, t) for a, t, s in zip(_ans, _t, _srcs) if s == "documentary_grounding"]
assert _doc and all(_ok_box(json.loads(a)["bbox_2d"]) for a, t in _doc if t == "locate"), "bad documentary box"
assert all(a.strip() for a, t in _doc if t != "locate"), "empty documentary read_box target"
_rb = [q for q, t in zip(_q, _t) if t in ("read_box", "read_box_word")]
assert _rb and all("bbox_2d = [" in q and "0-1000" in q for q in _rb), "read_box prompt drift"

# QA hygiene: every answer is {"line": N, "text": span} / a list of them / {"answer": "not stated"}
_qa = [(a, q, t) for a, q, t, s in zip(_ans, _q, _t, _srcs) if s == "pgp_qa"]
assert _qa, "no QA rows in the mixture"
for a, q, t in _qa:
    o = json.loads(a)
    if isinstance(o, dict) and "answer" in o:
        assert o["answer"] == "not stated" and t == "qa_abstain", a
    else:
        items = o if isinstance(o, list) else [o]
        assert items and all(isinstance(i["line"], int) and i["line"] >= 1 and i["text"].strip()
                             for i in items), a
    assert "JSON" in q, q
print(f"hygiene OK: transcription {len(_transc)}, locate {len(_loc)}, documentary {len(_doc)}, "
      f"read_box prompts {len(_rb)}, QA {len(_qa)} ({Counter(t for *_, t in _qa)})")


In [ ]:
# Cell 3 — model at page resolution + THE MERGER KNOB + flagship warm start (v2.1b step 1200)
from unsloth import FastVisionModel
from unsloth_zoo.peft_utils import get_peft_regex
from transformers import AutoImageProcessor

MAX_SEQ = 12288
# Global 6.5MP floor (train/infer resolution contract, ships in the export).
MIN_PIX = 6_500_000
MAX_PIX = 7_000_000

# ⬅️ THE DECISION KNOB — set per the v1.9c verdict:
#   "frozen" = v1.9a config (default, safe)   "full" = v1.9c full-weight merger
#   "lora"   = v1.9b merger LoRA (two-sided at r16 per the ablation; avoid)
MERGER_MODE = "frozen"
assert MERGER_MODE in ("frozen", "lora", "full")

# Warm start: v2.1b step 1200 — the flagship (religious CER 0.167, PGP 0.180, boxes on the right line 80%;
# the 1300–1700 checkpoints were never promoted). The v2.2 data lever sits on top of it.
WARM_CKPT_REPO = "isaacmg/qwen3-vl-8b-hebrew-v21b-ckpt"
WARM_REVISION = "6724c32cd0f6296223978ae69635fc782d2dbb03"  # commit "Training in progress, step 1200, checkpoint"

TRAIN_VISION_LORA = True    # unchanged since v1.8b
MERGER_MODULES = [
    "visual.merger.linear_fc1", "visual.merger.linear_fc2",
    *[f"visual.deepstack_merger_list.{i}.linear_fc{j}"
      for i in range(3) for j in (1, 2)],
]

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
    max_seq_length=MAX_SEQ,
)

base_regex = get_peft_regex(
    model,
    finetune_vision_layers=True, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
)
MERGER_REGEX = r".*\.visual\.(?:merger|deepstack_merger_list\.\d+)\.linear_fc[12]"
target_modules = (f"(?:{base_regex})|(?:{MERGER_REGEX})"
                  if MERGER_MODE == "lora" else base_regex)

if MERGER_MODE == "full":
    # modules_to_save cannot train a quantized clone (see v1.9c notebook)
    import bitsandbytes as bnb
    _mods = {n: m for n, m in model.named_modules()
             if any(n.endswith(s) for s in MERGER_MODULES)}
    assert len(_mods) == 8, f"merger modules found: {sorted(_mods)}"
    _quantized = [n for n, m in _mods.items()
                  if isinstance(m, (bnb.nn.Linear4bit, bnb.nn.Linear8bitLt))]
    assert not _quantized, (
        f"merger modules are quantized: {_quantized} — reload with these in "
        "llm_int8_skip_modules or load_in_4bit=False before proceeding.")

model = FastVisionModel.get_peft_model(
    model,
    target_modules=target_modules,
    modules_to_save=MERGER_MODULES if MERGER_MODE == "full" else None,
    finetune_vision_layers=True, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0.0, bias="none", random_state=3407,
)

# creation gate — the run is void unless the adapter matches the KNOB
_tower_lora = [n for n, _ in model.named_parameters()
               if ".visual.blocks." in n and "lora_A" in n]
_merger_lora = [n for n, _ in model.named_parameters()
                if "merger" in n and "lora_A" in n]
_merger_full = [n for n, p in model.named_parameters()
                if "merger" in n and ".modules_to_save." in n
                and n.endswith(".weight") and p.requires_grad]
assert len(_tower_lora) == 108, f"tower adapters: {len(_tower_lora)}/108"
if MERGER_MODE == "frozen":
    assert not _merger_lora and not _merger_full, "merger adapted but knob=frozen"
elif MERGER_MODE == "lora":
    assert len(_merger_lora) == 8 and not _merger_full,         f"merger lora {len(_merger_lora)}/8 — knob=lora not honoured"
else:
    assert len(_merger_full) == 8 and not _merger_lora,         f"merger clones {len(_merger_full)}/8 — knob=full not honoured"
print(f"MERGER_MODE={MERGER_MODE}: tower {len(_tower_lora)}/108, "
      f"merger lora {len(_merger_lora)}, merger full {len(_merger_full)}")

# weights-only warm start (fresh optimizer + schedule)
assert len(WARM_REVISION) == 40, "invalid revision SHA"
from safetensors.torch import load_file
from peft import set_peft_model_state_dict
local = snapshot_download(WARM_CKPT_REPO, revision=WARM_REVISION,
                          allow_patterns="last-checkpoint/adapter_model.safetensors")
_sd = load_file(f"{local}/last-checkpoint/adapter_model.safetensors")
_loaded_merger_lora = any("merger" in k and "lora" in k for k in _sd)
_loaded_merger_full = any("merger" in k and "lora" not in k for k in _sd)
# (modules_to_save saves PLAIN keys — the adapter-name infix is stripped on save)
# the knob must match what the warm checkpoint carries, or trained merger
# weights would be silently dropped on load
if _loaded_merger_full:
    assert MERGER_MODE == "full", "warm ckpt has full merger weights — set MERGER_MODE='full'"
if _loaded_merger_lora:
    assert MERGER_MODE == "lora", "warm ckpt has merger LoRA — set MERGER_MODE='lora'"
if MERGER_MODE == "full":
    # PEFT's modules_to_save loader KeyErrors when the incoming dict lacks the
    # merger keys (any pre-v1.9c warm start). Inject the base weights — the
    # identity start the gate below verifies; setdefault keeps warm-loaded
    # merger weights when the checkpoint carries them.
    for _n, _m in model.named_modules():
        if any(_n.endswith(s) for s in MERGER_MODULES) and hasattr(_m, "modules_to_save"):
            for _pn, _p in _m.original_module.named_parameters():
                _sd.setdefault(f"{_n}.{_pn}", _p.detach().clone())
missing = set_peft_model_state_dict(model, _sd)
print("unexpected keys:", len(getattr(missing, "unexpected_keys", [])))
_wv = [n for n, p in model.named_parameters()
       if ".visual.blocks." in n and "lora_B" in n and p.detach().abs().max().item() > 0]
assert len(_wv) == 108, f"warm start vision adapter not loaded: {len(_wv)}/108 nonzero"
if MERGER_MODE == "lora" and not _loaded_merger_lora:
    _mz = [n for n, p in model.named_parameters()
           if "merger" in n and "lora_B" in n and p.detach().abs().max().item() > 0]
    assert not _mz, f"merger lora_B nonzero at start: {_mz[:3]}"
if MERGER_MODE == "full":
    _wrappers = [(n, m) for n, m in model.named_modules()
                 if any(n.endswith(s) for s in MERGER_MODULES)
                 and hasattr(m, "modules_to_save")]
    assert len(_wrappers) == 8, f"modules_to_save wrappers: {len(_wrappers)}/8"
    if not _loaded_merger_full:
        for _n, _w in _wrappers:
            assert torch.allclose(
                _w.modules_to_save["default"].weight.detach().float(),
                _w.original_module.weight.detach().float()),                 f"merger clone differs from base at start: {_n}"
        print("merger clones at identity (fresh full-weight start)")
    else:
        print("merger clones warm-loaded from checkpoint")
print(f"warm start OK from {WARM_CKPT_REPO}@{WARM_REVISION[:8]}")

if TRAIN_VISION_LORA:
    # patch-embed require-grad fix (see v1.8b/v1.9 notebooks for the diagnosis)
    def _vision_embeds_require_grad(module, inputs, output):
        if not torch.is_grad_enabled():
            return output
        output.requires_grad_(True)
        return output
    _patch_embed = next(m for n, m in model.named_modules()
                        if n.endswith("visual.patch_embed"))
    _patch_embed.register_forward_hook(_vision_embeds_require_grad)
    print("vision LoRA gradient fix: ON")

tokenizer.image_processor = AutoImageProcessor.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct", min_pixels=MIN_PIX, max_pixels=MAX_PIX,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable: {trainable/1e6:.1f}M")
print("resolution:", tokenizer.image_processor.size)


In [ ]:
# Cell 4 — conversation format + collator (native res preserved by resize='max')
def to_conversation(sample):
    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "image", "image": sample["image"]},
                {"type": "text", "text": sample["question"]},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": sample["answer"]},
            ]},
        ]
    }

from unsloth.trainer import UnslothVisionDataCollator

collator = UnslothVisionDataCollator(
    model, tokenizer,
    formatting_func=to_conversation,
    resize="max",
    max_seq_length=MAX_SEQ,
    train_on_responses_only=True,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# guardrail: one real batch (a KTIV page + a QA row) must show page-res pixels + masked labels
_i_page = next(i for i, (t, s) in enumerate(zip(train_ds.column("task"), train_ds.column("source")))
               if t == "fragment_transcribe" and s.startswith("ktiv"))
_i_qa = next(i for i, s in enumerate(train_ds.column("source")) if s == "pgp_qa")
_page_row, _qa_row = train_ds[_i_page], train_ds[_i_qa]
batch = collator([_page_row, _qa_row])
pv = batch["pixel_values"]
assert pv is not None and pv.shape[0] > 40000, (
    f"{pv.shape[0]} patch rows — expected >40k for two ~6.5MP images; "
    "the resolution policy is not reaching the collator")
labels = batch["labels"]
_tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
for _k, _row in ((0, _page_row), (1, _qa_row)):
    _unmasked = labels[_k][labels[_k] != -100]
    assert _row["answer"][:30] in _tok.decode(_unmasked), f"row {_k}: answer not in the unmasked labels"
    assert _row["question"][:30] not in _tok.decode(_unmasked), f"row {_k}: prompt leaked into the loss"
print(f"collator OK: pixel rows={pv.shape[0]}, labels masked on both rows (page + QA)")


In [ ]:
# Cell 5 — gradient-flow verification (cheap; run BEFORE burning GPU-hours)
# Language LoRA always; tower LoRA iff TRAIN_VISION_LORA; merger per the KNOB.
FastVisionModel.for_training(model)
_vb = {k: (v.to(model.device) if torch.is_tensor(v) else v)
       for k, v in collator([train_ds[_i_page]]).items()}
model(**_vb).loss.backward()
tower_g = [p.grad.abs().max().item() for n, p in model.named_parameters()
           if ".visual.blocks." in n and "lora_B" in n and p.grad is not None]
mrg_lora_g = [p.grad.abs().max().item() for n, p in model.named_parameters()
              if "merger" in n and "lora_B" in n and p.grad is not None]
mrg_full_g = [p.grad.abs().max().item() for n, p in model.named_parameters()
              if "merger" in n and ".modules_to_save." in n and n.endswith(".weight")
              and p.grad is not None]
lang_g = [p.grad.abs().max().item() for n, p in model.named_parameters()
          if ".visual." not in n and "lora_B" in n and p.grad is not None]
model.zero_grad(set_to_none=True)
assert lang_g and max(lang_g) > 0, "language LoRA got no gradients"
if TRAIN_VISION_LORA:
    assert len(tower_g) == 108 and min(tower_g) > 0, (
        f"tower LoRA dead or partial: {len(tower_g)}/108 tensors with grads")
if MERGER_MODE == "lora":
    assert len(mrg_lora_g) == 8 and min(mrg_lora_g) > 0,         f"merger LoRA dead or partial: {len(mrg_lora_g)}/8"
elif MERGER_MODE == "full":
    assert len(mrg_full_g) == 8 and min(mrg_full_g) > 0,         f"merger full weights dead or partial: {len(mrg_full_g)}/8"
else:
    assert not mrg_lora_g and not mrg_full_g, "merger got grads but knob=frozen"
print(f"grad gate OK (MERGER_MODE={MERGER_MODE}): tower min|g|={min(tower_g):.2e} "
      f"language max|g|={max(lang_g):.2e}")


In [ ]:
# Cell 6a — OPTIONAL throughput benchmark (flag-gated; run once, then set False)
# The A100 ran v18 at batch 1x8 with 13.8/40GB VRAM — headroom. This times
# fwd+bwd for candidate batch geometries at the REAL resolution so the long
# run uses the fastest safe config. To compare gradient-checkpointing modes
# ("unsloth" vs True), change it in cell 3 and rerun cells 3-6a once each.
RUN_THROUGHPUT_BENCH = False
BATCH_GEOMETRIES = [(1, 8), (2, 4), (4, 2)]   # (per_device_batch, grad_accum)

if RUN_THROUGHPUT_BENCH:
    import time
    FastVisionModel.for_training(model)
    bench_rows = [train_ds[i] for i in range(16)]   # map-style mixture: first 16 rows (all sources)
    for bs, ga in BATCH_GEOMETRIES:
        try:
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            t0 = time.time(); n_micro = 0
            for step in range(2):                    # 2 optimizer steps
                for micro in range(ga):
                    rows = [bench_rows[(n_micro + i) % len(bench_rows)] for i in range(bs)]
                    b = {k: (v.to(model.device) if torch.is_tensor(v) else v)
                         for k, v in collator(rows).items()}
                    (model(**b).loss / ga).backward()
                    n_micro += 1
                model.zero_grad(set_to_none=True)
            dt = (time.time() - t0) / 2
            peak = torch.cuda.max_memory_allocated() / 1e9
            print(f"batch {bs}x{ga}: {dt:.1f}s/optimizer-step, peak {peak:.1f}GB")
        except torch.cuda.OutOfMemoryError:
            print(f"batch {bs}x{ga}: OOM — skip")
            torch.cuda.empty_cache()
    model.zero_grad(set_to_none=True)
    print("pick the fastest non-OOM geometry and set it in cell 6, then set "
          "RUN_THROUGHPUT_BENCH = False")

In [ ]:
# Cell 6 — v2.2a training: materialised map-style mixture + per-source eval
import inspect
import wandb
from huggingface_hub import list_repo_files
from torch.utils.data import Dataset as _TDataset
from trl import SFTConfig, SFTTrainer
from unsloth import is_bf16_supported

CKPT_REPO = "isaacmg/qwen3-vl-8b-hebrew-v22a-ckpt"  # NEW repo — never reuse v21b's (auto-resume would load its last-checkpoint)
OUT_DIR = "outputs_v22a"
wandb.login(key=userdata.get("WANDB_API_KEY"))

def make_sft_config(**kw):
    params = inspect.signature(SFTConfig.__init__).parameters
    if "max_seq_length" in kw and "max_seq_length" not in params:
        kw["max_length"] = kw.pop("max_seq_length")
    dropped = {k: kw.pop(k) for k in list(kw) if k not in params}
    if dropped:
        print(f"⚠️ dropped unsupported SFTConfig kwargs: {sorted(dropped)}")
    return SFTConfig(**kw)

def make_trainer(**kw):
    try:
        return SFTTrainer(**kw)
    except TypeError as e:
        if "tokenizer" in kw and ("tokenizer" in str(e) or "processing_class" in str(e)):
            kw["processing_class"] = kw.pop("tokenizer")
            return SFTTrainer(**kw)
        raise

resume_dir = None
try:
    # hub_strategy="checkpoint" pushes a rolling "last-checkpoint/" folder
    files = list_repo_files(CKPT_REPO)
    if any(f.startswith("last-checkpoint/") for f in files):
        snapshot_download(CKPT_REPO, allow_patterns="last-checkpoint/*", local_dir=OUT_DIR)
        resume_dir = f"{OUT_DIR}/last-checkpoint"
        print("resuming from last-checkpoint")
except Exception as e:
    print(f"no checkpoint repo yet ({type(e).__name__}) — fresh start")

FastVisionModel.for_training(model)
trainer = make_trainer(
    model=model, tokenizer=tokenizer, data_collator=collator,
    train_dataset=train_ds, eval_dataset=eval_ds,
    args=make_sft_config(
        per_device_train_batch_size=1, gradient_accumulation_steps=8,   # 1x8 proven on the A100 (~16/40GB, ~65 s/step)
        dataloader_num_workers=2, dataloader_pin_memory=True,
        max_steps=2000,                    # 8,000 rows/epoch ⇒ 1,000 steps/epoch; kill-anytime, resumes
        learning_rate=5e-5,
        warmup_ratio=0.02, lr_scheduler_type="cosine", weight_decay=0.01,
        logging_steps=10,
        eval_strategy="steps", eval_steps=100, per_device_eval_batch_size=1,
        save_steps=100, save_total_limit=2,
        push_to_hub=True, hub_model_id=CKPT_REPO,
        hub_strategy="checkpoint", hub_private_repo=False,
        optim="adamw_8bit", seed=3407, data_seed=3407, output_dir=OUT_DIR,
        report_to="wandb", run_name="genizah_v22a",
        bf16=is_bf16_supported(), fp16=not is_bf16_supported(),
        remove_unused_columns=False, dataset_text_field="",
        # map-style train set: Accelerate does not dispatch, but the v2.1 rule stays explicit and gated
        accelerator_config={"dispatch_batches": False},
        dataset_kwargs={"skip_prepare_dataset": True},
        max_seq_length=MAX_SEQ,
    ),
)
wandb.config.update({"v22_repo": V22_REPO, "v22_revision": V22_REVISION, "warm_ckpt": f"{WARM_CKPT_REPO}@{WARM_REVISION}",
                     "mixture": MIX.get("config", {})}, allow_val_change=True)

# ---- DATALOADER GATE (v2.1 lesson, kept for map-style) — checks the PREPARED train dataloader, not the collator ----
assert trainer.args.accelerator_config.dispatch_batches is False, "dispatch_batches=False did not reach the trainer"

class _Recording(_TDataset):
    """Wrapper that records fetched indices so the gate can find the row behind a prepared batch."""
    def __init__(self, ds):
        self.ds, self.seen = ds, []
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        self.seen.append(int(i))
        return self.ds[i]

_rec = _Recording(train_ds)
trainer.train_dataset = _rec
_saved_workers, trainer.args.dataloader_num_workers = trainer.args.dataloader_num_workers, 0  # in-process fetch ⇒ indices visible
_dl = trainer.get_train_dataloader()
assert "Dispatcher" not in type(_dl).__name__, f"train dataloader is {type(_dl).__name__} — batches would be truncated"
_it = iter(_dl)
for _k in range(2):
    _b = next(_it)
    _need = int(_b["image_grid_thw"].prod(dim=-1).sum())
    assert _b["pixel_values"].shape[0] == _need, (
        f"batch {_k}: pixel_values has {_b['pixel_values'].shape[0]} patch rows, grid needs {_need} — truncated")
    if _k == 0:
        _first, _idx = _b, _rec.seen[0]
# the loss on a prepared batch must equal the loss on the same row collated directly (image intact end to end)
_row = train_ds[_idx]
_direct = {k: (v.to(model.device) if torch.is_tensor(v) else v) for k, v in collator([_row]).items()}
assert torch.equal(_direct["input_ids"], _first["input_ids"].to(model.device)), (
    f"prepared batch 0 is not train row {_idx} — the recording wrapper and the sampler disagree")
model.eval()
with torch.no_grad():
    _lp = model(**{k: (v.to(model.device) if torch.is_tensor(v) else v) for k, v in _first.items()}).loss.item()
    _ld = model(**_direct).loss.item()
assert abs(_lp - _ld) <= 0.05 * max(1.0, _ld), f"prepared-dataloader loss {_lp:.3f} != direct-collate loss {_ld:.3f}"
_eb = next(iter(trainer.get_eval_dataloader()))
assert _eb["pixel_values"].shape[0] == int(_eb["image_grid_thw"].prod(dim=-1).sum()), "eval batch truncated"
print(f"dataloader gate OK: {type(_dl).__name__}, patches intact, row {_idx} ({_row['source']}/{_row['task']}), "
      f"loss prepared={_lp:.3f} direct={_ld:.3f}")
trainer.train_dataset = train_ds
trainer.args.dataloader_num_workers = _saved_workers
del _dl, _it, _b, _first, _row, _direct, _eb, _rec
FastVisionModel.for_training(model)
if resume_dir:
    # map-style resume: the Trainer would replay (collate) every seen batch on the CPU to skip it —
    # hours of idle GPU for nothing (v2.1b lesson). Restore optimizer/scheduler/step and start a fresh
    # shuffled epoch instead; a few rows repeat or are skipped, which is fine for a fine-tune.
    trainer.args.ignore_data_skip = True
    print("resume: optimizer/scheduler/step restored, data order restarted (ignore_data_skip=True)")
trainer.train(resume_from_checkpoint=resume_dir)


In [ ]:
# Cell 7 — export merged (policy-carrying) model, gated per the KNOB
MERGED_REPO = "isaacmg/qwen3-vl-8b-hebrew-v22a-merged"
if TRAIN_VISION_LORA:
    _nz = [n for n, p in model.named_parameters()
           if ".visual.blocks." in n and "lora_B" in n and p.detach().abs().max().item() > 0]
    assert len(_nz) == 108, f"shipped adapter tower lora_B nonzero: {len(_nz)}/108"
if MERGER_MODE == "lora":
    _mnz = [n for n, p in model.named_parameters()
            if "merger" in n and "lora_B" in n and p.detach().abs().max().item() > 0]
    assert len(_mnz) == 8, f"merger lora_B nonzero: {len(_mnz)}/8 — merger never moved"
elif MERGER_MODE == "full":
    _moved = []
    for _n, _m in model.named_modules():
        if any(_n.endswith(s) for s in MERGER_MODULES) and hasattr(_m, "modules_to_save"):
            _c = _m.modules_to_save["default"].weight.detach().float()
            _o = _m.original_module.weight.detach().float()
            if not torch.allclose(_c, _o):
                _moved.append(_n)
    assert len(_moved) == 8, f"merger full weights moved: {len(_moved)}/8"
print(f"shipped checks OK (MERGER_MODE={MERGER_MODE})")
model.save_pretrained_merged("v22a-merged", tokenizer, save_method="merged_16bit")
# ship the TRAINING resolution policy with the model (hard rule)
tokenizer.image_processor.save_pretrained("v22a-merged")
model.push_to_hub_merged(MERGED_REPO, tokenizer, save_method="merged_16bit", private=True)
print("pushed", MERGED_REPO)
